In [ ]:
import pandas as pd

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()

# List datasets in your project
datasets = list(client.list_datasets())

if datasets:
    print("Datasets in project:")
    for dataset in datasets:
        print(f"{dataset.dataset_id}")
else:
    print("No datasets found.")

No datasets found.


In [ ]:
print(f"Default Project: {client.project}")


Default Project: trim-mariner-450817-p7


In [ ]:
from google.cloud import bigquery

# Initialize BigQuery client
client = bigquery.Client()

# Set dataset details
project_id = client.project  # Uses your default project
dataset_id = "my_dataset"    # Change this name if needed

# Construct a full Dataset ID
full_dataset_id = f"{project_id}.{dataset_id}"

# Create dataset if it doesn't exist
dataset = bigquery.Dataset(full_dataset_id)
dataset.location = "US"  # Set location (choose as per your region, e.g., US, EU)

dataset = client.create_dataset(dataset, exists_ok=True)
print(f"Dataset '{full_dataset_id}' created or already exists.")

Dataset 'trim-mariner-450817-p7.my_dataset' created or already exists.


In [ ]:
datasets = list(client.list_datasets())
print("Datasets in project:")
for dataset in datasets:
    print(dataset.dataset_id)


Datasets in project:
my_dataset


In [ ]:
#This is for Review table
# Set table name
table_id = f"{project_id}.{dataset_id}.my_table"

# JSON file from GCS
gcs_uri = "gs://unt_capstone_project_2025/yelp_academic_dataset_review.json"  # Change this to your actual GCS path

# Table configuration
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    autodetect=True,
)

# Load data from GCS into BigQuery
load_job = client.load_table_from_uri(
    gcs_uri,
    table_id,
    job_config=job_config
)

load_job.result()  # Wait for the job to complete
print(f"Table '{table_id}' created with data from {gcs_uri}.")


Table 'trim-mariner-450817-p7.my_dataset.my_table' created with data from gs://unt_capstone_project_2025/yelp_academic_dataset_review.json.


In [ ]:
query = f"""
    SELECT *
    FROM `{project_id}.{dataset_id}.my_table`
"""
review_df = client.query(query).to_dataframe()
review_df.head()


,text,cool,stars,date,funny,review_id,useful,business_id,user_id
0,ChopHouse\n\nWith everything going on in the w...,4,5.0,2021-07-08 02:31:34+00:00,1,iMaxU340B3Gz43hAsaDmRw,6,iC9Gis3-VspIr8Ox3e2beA,p951o8o4e3cHC1DSyiKP7Q
1,Another spot that just doesn't disappoint! The...,0,5.0,2019-05-16 03:36:13+00:00,0,hl4PnDzKGaF8xTwStIfDXw,0,M-6AXSgmDMYcWsF442mzzA,gFFMMMlnoqt-3YkfhaaI9Q
2,"As a same day wedding coordinator, this shop i...",0,5.0,2017-02-07 19:22:29+00:00,0,Afs4ChtZB4yf6yapfoo0Hg,0,Rnvl_hNexRJvC9zwVzPOdw,xShyBuTNL2mFZyvkL0vtPQ
3,This was possibly the best Mexican meal I've e...,0,5.0,2018-09-16 14:06:17+00:00,0,wZgsjr1hA5HJ189Bx8FPfQ,0,1Ly-Njk6U0kEq3TFnCgeWw,MTPka8o3xwDpEMMGQBtg5g
4,"Had a head unit, amplifier and a set of compon...",0,5.0,2020-09-15 21:48:29+00:00,0,Ac2n_YmBBeIEcRrOYOWZOg,0,h_vx0fR8lnwsrJASl_S8Aw,AgAqKRshVVuQQdxVw9zMxA


In [ ]:
review_df.shape

(6990280, 9)

In [ ]:
review_df['date'].max()

Timestamp('2022-01-19 19:48:45+0000', tz='UTC')

In [ ]:
review_df_subset = review_df[review_df['date'] > '2016-12-31']

In [ ]:
review_df_subset.to_parquet('gs://unt_capstone_project_2025/review_subset.parquet')

In [ ]:
review_df_subset.shape

(3840621, 9)